# fasttext

In [1]:
import pandas as pd
import pickle

In [2]:
grouped = pd.read_pickle('/srv/data/enemy-christ/large-data/grouped_df.pkl')

In [3]:
# make your subcorpora dataframes from "enemy_subcorpus" column
christian_0_300 = grouped[grouped["enemy_subcorpus"]=="christian_0_300"]
christian_300_600 = grouped[grouped["enemy_subcorpus"]=="christian_300_600"]
pagan_0_300 = grouped[grouped["enemy_subcorpus"]=="pagan_0_300"]
pagan_300_600 = grouped[grouped["enemy_subcorpus"]=="pagan_300_600"]

In [4]:
from nltk.probability import FreqDist

fdist1 = FreqDist(word for sent in christian_0_300['lemma_sentence'] for word in sent)
fdist2 = FreqDist(word for sent in christian_300_600['lemma_sentence'] for word in sent)
fdist3 = FreqDist(word for sent in pagan_0_300['lemma_sentence'] for word in sent)
fdist4 = FreqDist(word for sent in pagan_300_600['lemma_sentence'] for word in sent)

# Find words that appear >=10 times in all subcorpora
common_words = set(
    word for word in fdist1 if fdist1[word] >= 10
) & set(
    word for word in fdist2 if fdist2[word] >= 10
) & set(
    word for word in fdist3 if fdist3[word] >= 10
) & set(
    word for word in fdist4 if fdist4[word] >= 10
)

# Filter each subcorpus to only keep those words
words1 = {word: fdist1[word] for word in common_words}
words2 = {word: fdist2[word] for word in common_words}
words3 = {word: fdist3[word] for word in common_words}
words4 = {word: fdist4[word] for word in common_words}

In [5]:
from gensim.models import FastText
from gensim.models.callbacks import CallbackAny2Vec
from gensim.utils import RULE_KEEP, RULE_DISCARD, RULE_DEFAULT

class EpochLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0
    def on_epoch_begin(self, model):
        print(f"Epoch #{self.epoch} start")
    def on_epoch_end(self, model):
        print(f"Epoch #{self.epoch} end")
        self.epoch += 1

def train_fasttext_on_subcorpus(sentences, model_path):
    #sentences = [s.split() for s in sentences]
    model = FastText(
        vector_size=100,
        window=10, #vojta used 10
        min_count=1,
        sg=1,
        workers=16,
    )
    model.build_vocab_from_freq(word_freq=sentences)
    model.train(
        corpus_iterable=sentences,
        total_examples=len(sentences),
        epochs=20,
        callbacks=[EpochLogger()]
    )
    model.save(model_path)
    return model


In [6]:
%%time
# Example usage for each subcorpus:
christian_0_300_model = train_fasttext_on_subcorpus(
    words1, "../data/large-data/fasttext_christian_0_300.model"
)
christian_300_600_model = train_fasttext_on_subcorpus(
    words2, "../data/large-data/fasttext_christian_300_600.model"
)
pagan_0_300_model = train_fasttext_on_subcorpus(
    words3, "../data/large-data/fasttext_pagan_0_300.model"
)
pagan_300_600_model = train_fasttext_on_subcorpus(
    words4, "../data/large-data/fasttext_pagan_300_600.model"
)

Epoch #0 start
Epoch #0 end
Epoch #1 start
Epoch #1 end
Epoch #2 start
Epoch #2 end
Epoch #3 start
Epoch #3 end
Epoch #4 start
Epoch #4 end
Epoch #5 start
Epoch #5 end
Epoch #6 start
Epoch #6 end
Epoch #7 start
Epoch #7 end
Epoch #8 start
Epoch #8 end
Epoch #9 start
Epoch #9 end
Epoch #10 start
Epoch #10 end
Epoch #11 start
Epoch #11 end
Epoch #12 start
Epoch #12 end
Epoch #13 start
Epoch #13 end
Epoch #14 start
Epoch #14 end
Epoch #15 start
Epoch #15 end
Epoch #16 start
Epoch #16 end
Epoch #17 start
Epoch #17 end
Epoch #18 start
Epoch #18 end
Epoch #19 start
Epoch #19 end
Epoch #0 start
Epoch #0 end
Epoch #1 start
Epoch #1 end
Epoch #2 start
Epoch #2 end
Epoch #3 start
Epoch #3 end
Epoch #4 start
Epoch #4 end
Epoch #5 start
Epoch #5 end
Epoch #6 start
Epoch #6 end
Epoch #7 start
Epoch #7 end
Epoch #8 start
Epoch #8 end
Epoch #9 start
Epoch #9 end
Epoch #10 start
Epoch #10 end
Epoch #11 start
Epoch #11 end
Epoch #12 start
Epoch #12 end
Epoch #13 start
Epoch #13 end
Epoch #14 start
Epoc

In [7]:
from gensim.models import FastText

# Load every FastText model for each subcorpus into a variable
christian_0_300_model = FastText.load("../data/large-data/fasttext_christian_0_300.model")
christian_300_600_model = FastText.load("../data/large-data/fasttext_christian_300_600.model")
pagan_0_300_model = FastText.load("../data/large-data/fasttext_pagan_0_300.model")
pagan_300_600_model = FastText.load("../data/large-data/fasttext_pagan_300_600.model")

In [8]:
%%time
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE

models = [
    (christian_0_300_model, "christian_0_300", 0),
    (christian_300_600_model, "christian_300_600", 1),
    (pagan_0_300_model, "pagan_0_300", 2),
    (pagan_300_600_model, "pagan_300_600", 3),
]

dfs = []
for model, label, z in models:
    vocab = list(model.wv.index_to_key)
    vecs = [model.wv[w] for w in vocab]
    tsne = TSNE(n_components=2, random_state=42, verbose=1)
    tsne_result = tsne.fit_transform(np.array(vecs))
    df = pd.DataFrame({
        "word": vocab,
        "x": tsne_result[:, 0],
        "y": tsne_result[:, 1],
        "z": z,
        "subcorpus": label
    })
    dfs.append(df)

df_plot = pd.concat(dfs, ignore_index=True)
df_plot.to_pickle("../data/large-data/df_plot.pkl")

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 3770 samples in 0.001s...
[t-SNE] Computed neighbors for 3770 samples in 0.084s...
[t-SNE] Computed conditional probabilities for sample 1000 / 3770
[t-SNE] Computed conditional probabilities for sample 2000 / 3770
[t-SNE] Computed conditional probabilities for sample 3000 / 3770
[t-SNE] Computed conditional probabilities for sample 3770 / 3770
[t-SNE] Mean sigma: 0.003306
[t-SNE] KL divergence after 250 iterations with early exaggeration: 83.116318
[t-SNE] KL divergence after 1000 iterations: 3.245912
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 3770 samples in 0.000s...
[t-SNE] Computed neighbors for 3770 samples in 0.038s...
[t-SNE] Computed conditional probabilities for sample 1000 / 3770
[t-SNE] Computed conditional probabilities for sample 2000 / 3770
[t-SNE] Computed conditional probabilities for sample 3000 / 3770
[t-SNE] Computed conditional probabilities for sample 3770 / 3770
[t-SNE] Mean sigma: 0.003302


In [9]:
%%time
# Precompute nearest neighbors for all words in all subcorpora and save to pickle
from collections import defaultdict

neighbors_dict = defaultdict(dict)
model_map = {
    "christian_0_300": christian_0_300_model,
    "christian_300_600": christian_300_600_model,
    "pagan_0_300": pagan_0_300_model,
    "pagan_300_600": pagan_300_600_model,
}
N_NEIGHBORS = 100

for subcorpus, model in model_map.items():
    for word in model.wv.index_to_key:
        try:
            neighbors = model.wv.most_similar(word, topn=N_NEIGHBORS)  # returns list of (word, similarity)
        except KeyError:
            neighbors = []
        neighbors_dict[subcorpus][word] = neighbors

import pickle
with open("../data/large-data/word_neighbors.pkl", "wb") as f:
    pickle.dump(dict(neighbors_dict), f)

CPU times: user 23 s, sys: 527 ms, total: 23.6 s
Wall time: 1.68 s


# TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.probability import FreqDist
import pandas as pd
import numpy as np
import pickle

In [ ]:
grouped = pd.read_pickle('/srv/data/enemy-christ/large-data/grouped_df.pkl')

In [10]:
# make your subcorpora dataframes from "enemy_subcorpus" column
christian_0_300 = grouped[grouped["enemy_subcorpus"]=="christian_0_300"]
christian_300_600 = grouped[grouped["enemy_subcorpus"]=="christian_300_600"]
pagan_0_300 = grouped[grouped["enemy_subcorpus"]=="pagan_0_300"]
pagan_300_600 = grouped[grouped["enemy_subcorpus"]=="pagan_300_600"]

In [ ]:
# Find most frequent words (lowercased)
fdist1 = FreqDist(word.lower() for sent in christian_0_300['lemma_sentence'] for word in sent)
fdist2 = FreqDist(word.lower() for sent in christian_300_600['lemma_sentence'] for word in sent)
fdist3 = FreqDist(word.lower() for sent in pagan_0_300['lemma_sentence'] for word in sent)
fdist4 = FreqDist(word.lower() for sent in pagan_300_600['lemma_sentence'] for word in sent)

# Find words that appear >=10 times in all subcorpora
vocab = set(
    word for word in fdist1 if fdist1[word] >= 10
) & set(
    word for word in fdist2 if fdist2[word] >= 10
) & set(
    word for word in fdist3 if fdist3[word] >= 10
) & set(
    word for word in fdist4 if fdist4[word] >= 10
)

In [ ]:
def compute_tfidf(sentences):
    docs = [" ".join(s) if isinstance(s, (list, tuple)) else s for s in sentences]
    vectorizer = TfidfVectorizer(
        #min_df=10,
        #max_features=1000,
        vocabulary=vocab, #use common vocab
    )
    tfidf_matrix = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    return [tfidf_matrix, feature_names]

In [13]:
tfidf_christian_0_300 = compute_tfidf(christian_0_300['lemma_sentence'])
tfidf_christian_300_600 = compute_tfidf(christian_300_600['lemma_sentence'])
tfidf_pagan_0_300 = compute_tfidf(pagan_0_300['lemma_sentence'])
tfidf_pagan_300_600 = compute_tfidf(pagan_300_600['lemma_sentence'])

In [ ]:
# Compute co-occurrence matrices for all subcorpora
cooc_christian_0_300 = compute_weighted_cooccurrence(tfidf_christian_0_300)
cooc_christian_300_600 = compute_weighted_cooccurrence(tfidf_christian_300_600)
cooc_pagan_0_300 = compute_weighted_cooccurrence(tfidf_pagan_0_300)
cooc_pagan_300_600 = compute_weighted_cooccurrence(tfidf_pagan_300_600)

tfidf_cooc = {
    "christian_0_300": cooc_christian_0_300,
    "christian_300_600": cooc_christian_300_600,
    "pagan_0_300": cooc_pagan_0_300,
    "pagan_300_600": cooc_pagan_300_600,
}

with open("../data/large-data/tfidf_cooc.pkl", "wb") as f:
    pickle.dump(tfidf_cooc, f)

In [ ]:
def strongest_cooccurrences_per_subcorpus(tfidf_cooc, target_word, top_n=10):
    rows = []
    for subcorpus, cooc_df in tfidf_cooc.items():
        if target_word not in cooc_df.columns:
            continue
        scores = cooc_df[target_word].drop(labels=[target_word], errors='ignore').sort_values(ascending=False).head(top_n)
        for word, value in scores.items():
            rows.append({
                'subcorpus': subcorpus,
                'word': word,
                'tf-idf_score': value
            })
    return pd.DataFrame(rows, columns=['word', 'tf-idf_score', 'subcorpus'])

# usage example
tf_table = strongest_cooccurrences_per_subcorpus(tfidf_cooc, "πλανάω", top_n=10)
tf_table.sample(10)

# PMI

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.probability import FreqDist
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations

In [ ]:
grouped = pd.read_pickle('/srv/data/enemy-christ/large-data/grouped_df.pkl')

In [11]:
# make your subcorpora dataframes from "enemy_subcorpus" column
christian_0_300 = grouped[grouped["enemy_subcorpus"]=="christian_0_300"]
christian_300_600 = grouped[grouped["enemy_subcorpus"]=="christian_300_600"]
pagan_0_300 = grouped[grouped["enemy_subcorpus"]=="pagan_0_300"]
pagan_300_600 = grouped[grouped["enemy_subcorpus"]=="pagan_300_600"]

In [ ]:
# Find most frequent words (lowercased)
fdist1 = FreqDist(word.lower() for sent in christian_0_300['lemma_sentence'] for word in sent)
fdist2 = FreqDist(word.lower() for sent in christian_300_600['lemma_sentence'] for word in sent)
fdist3 = FreqDist(word.lower() for sent in pagan_0_300['lemma_sentence'] for word in sent)
fdist4 = FreqDist(word.lower() for sent in pagan_300_600['lemma_sentence'] for word in sent)

# Find words that appear >=10 times in all subcorpora
vocab = set(
    word for word in fdist1 if fdist1[word] >= 10
) & set(
    word for word in fdist2 if fdist2[word] >= 10
) & set(
    word for word in fdist3 if fdist3[word] >= 10
) & set(
    word for word in fdist4 if fdist4[word] >= 10
)

In [ ]:
def compute_pmi_from_sentences(sentences, vocab=vocab, min_count=10):
    # Tokenize sentences
    tokenized = [sent for sent in sentences]
    # Flatten for unigram counts
    all_words = [w for sent in tokenized for w in sent]
    unigram_counts = Counter(all_words)
    # Filter vocab by min_count
    if vocab is None:
        vocab = set([w for w, c in unigram_counts.items() if c >= min_count])
    # Count co-occurrences (within sentence)
    cooc_counts = Counter()
    for sent in tokenized:
        words = [w for w in sent if w in vocab]
        for w1, w2 in combinations(sorted(set(words)), 2):
            cooc_counts[(w1, w2)] += 1
    N = len(tokenized)
    results = []
    for (w1, w2), c_xy in cooc_counts.items():
        c_x = unigram_counts[w1]
        c_y = unigram_counts[w2]
        # Probabilities
        p_x = c_x / N
        p_y = c_y / N
        p_xy = c_xy / N
        if p_xy > 0:
            pmi = np.log(p_xy / (p_x * p_y))
            results.append({'word1': w1, 'word2': w2, 'pmi': pmi, 'count': c_xy})
    return pd.DataFrame(results)

In [ ]:
pmi_christian_0_300 = compute_pmi_from_sentences(christian_0_300['lemma_sentence'])
pmi_christian_300_600 = compute_pmi_from_sentences(christian_300_600['lemma_sentence'])
pmi_pagan_0_300 = compute_pmi_from_sentences(pagan_0_300['lemma_sentence'])
pmi_pagan_300_600 = compute_pmi_from_sentences(pagan_300_600['lemma_sentence'])

pmi_dict = {
    "christian_0_300": pmi_christian_0_300,
    "christian_300_600": pmi_christian_300_600,
    "pagan_0_300": pmi_pagan_0_300,
    "pagan_300_600": pmi_pagan_300_600,
}

with open("../data/large-data/pmi_all_subcorpora.pkl", "wb") as f:
    pickle.dump(pmi_dict, f)

In [ ]:
with open("../data/large-data/pmi_all_subcorpora.pkl", "rb") as f:
    pmi_dict = pickle.load(f)

def top_cooccurrences_per_subcorpus(pmi_dict, target_word, top_n=10):
    result = []
    for subcorpus, df in pmi_dict.items():
        if df is None or df.empty:
            continue
        mask = (df['word1'] == target_word) | (df['word2'] == target_word)
        sel = df[mask].copy()
        if sel.empty:
            continue
        sel['word'] = sel.apply(lambda r: r['word2'] if r['word1'] == target_word else r['word1'], axis=1)
        sel = sel[['word', 'pmi', 'count']].copy()
        sel['subcorpus'] = subcorpus
        sel = sel.sort_values(by='pmi', ascending=False).head(top_n)
        result.append(sel)
    if not result:
        return pd.DataFrame(columns=['word', 'pmi', 'count', 'subcorpus'])
    return pd.concat(result, ignore_index=True)[['word', 'pmi', 'count', 'subcorpus']]

# example usage
pmi_table = top_cooccurrences_per_subcorpus(pmi_dict, "πλανάω", top_n=10)
pmi_table.sample(10)